# DSCI 522


## Lecture 8 - Environment management: containerization with Docker


### 2017-12-12

# Learning Objectives

## By the end of the lecture, students should be able to:

- Write a Dockerfile
- Install R packages inside a Dockerfile
- Build a Docker image from a Dockerfile using trusted builds on Docker Hub
- List good Docker base images for R and Python projects

## Running rocker Docker images from the Shell instead of RStudio

The default behaviour for the rocker Docker images, including the `rocker/hadleyverse` image is to launch RStudio. You can override this as follows:

```
docker run --rm -it --user rstudio rocker/hadleyverse /bin/bash

```

The key is to add `/bin/bash` after the you call the image. This should also work for other images that have other default start-up behaviours.

## Installing R packages inside a Dockerfile

Technically, there are several ways to install R packages via a Dockerfile, but here is our recommended practice:

```
RUN R -e "install.packages('gapminder', repos='http://cran.us.r-project.org')"
```

The advantage of this strategy is that it automatically installs any R package depenedencies.

## Managing R package versions inside Docker

If your R project uses `packrat`, you can use `packrat` to manage the package versions. Another possibility is to use checkpoint/mran/snapshots from CRAN to lock-in the version:

```
RUN R -e "install.packages('gapminder', repos = 'https://mran.revolutionanalytics.com/snapshot/2016-10-19')"
```

*note - if you use this method, there can be problems with package dependency auto-installation, so use with caution...*

## Installing Python packages inside a Dockerfile

For non-complicated packages try:

```
RUN pip3 install numpy
```

If this doesn't work, then you may have to look for examples and/or experiment.

## Example Dockerfile for [`data_analysis_pipeline_eg`](https://github.com/ttimbers/data_analysis_pipeline_eg)

To create a `Dockerfile` that we can use to build the environment to run our data analysis pipeline in [`data_analysis_pipeline_eg`](https://github.com/ttimbers/data_analysis_pipeline_eg) we started with the `rocker/tidyverse` image and then installed the following things:

- `ezknitr` (R package)
- Python 3
- some Python dependencies (e.g., `pip3` and `python3-tk`)
- `numpy` (Python package)
- `matplotlib` (Python package)

#### Dockerfile:

```
# Docker file for data_analysis_pipeline_eg
# Tiffany Timbers, Dec, 2017

# use rocker/tidyverse as the base image and
FROM rocker/tidyverse

# then install the ezknitr packages
RUN Rscript -e "install.packages('ezknitr', repos = 'http://cran.us.r-project.org')"

# install python 3
RUN apt-get update \
  && apt-get install -y python3-pip python3-dev \
  && cd /usr/local/bin \
  && ln -s /usr/bin/python3 python \
  && pip3 install --upgrade pip

# get python package dependencies
RUN apt-get install -y python3-tk

# install numpy & matplotlib
RUN pip3 install numpy
RUN apt-get update && \
    pip3 install matplotlib && \
    rm -rf /var/lib/apt/lists/*
```

## Where does the Dockerfile live?

The Dockerfile should live in the root directory of your project.

## How do I make a Docker image from a Dockerfile?

There are 2 ways to do this! I use the first when developing my Dockerfile (to test quickly that it works), and then the second I use when I think I am "done" and want to have it archived on [Docker Hub](https://hub.docker.com/). 

A. `docker build`

    From the directory that contains your Dockerfile (usually your project root):
    ```
    docker build --tag data_analysis_pipeline_eg .
    ```
    
    *note: `--tag` let's you name the Docker image. You can call this anything you want.*
    
    After I build, I think try to `docker run ...` to test the image locally. If I don't like it, or it doesn't work, I delete the image with `docker rmi {IMAGE_NAME}`, edit my Dockerfile and try to build and run it again.


B. [Automated builds](https://docs.docker.com/docker-cloud/builds/automated-build/) on [Docker Hub](https://hub.docker.com/)

  1. Sign-up for an account on [Docker Hub](https://hub.docker.com/)
  2. Select "Create" > "Create an Automated build" and follow the instructions (you will be asked to link your Github account to your Docker Hub account).
  3. Go to the Docker Hub repository you created (by following the "Create an Automated build" instructions) and click on "Build Settings", and on that page, click "Trigger"
  4. After several minutes, your Docker Hub repository should say "AUTOMATED BUILD" if it successfully built and you should be able to view the Dockerfile on Docker Hub as well. 
  5. To verify, delete the local image on your computer and try pulling it from Docker Hub (you will need to include your username, for example: ttimbers/data_analysis_eg).
    

## Good base images to work from for R or Python projects!

| Image | Software installed | 
|-------|--------------------|
| [rocker/hadleyverse](https://hub.docker.com/r/rocker/hadleyverse/) | R, R packages (including the tidyverse), RStudio, make |
| [continuumio/anaconda3](https://hub.docker.com/r/continuumio/anaconda3/) | Python 3.5, Jupyter notebook, make(?) |

*note - currently there is a flaw in the [continuumio/anaconda3](https://hub.docker.com/r/continuumio/anaconda3/) Dockerfile and I keep getting errors with `matplotlib`... Others have documented this too... So maybe it will be more useful in the future?*

## Running `make` non-interactively & from a different directory through your Dockerfile

If we run our `data_analysis_pipeline_eg` container and link it to our `data_analysis_pipeline_eg` directory by:

```
docker run --rm -it -v /Users/timberst/Documents/ubc-mds-2017/data_analysis_pipeline_eg:/home/data_analysis_eg data_analysis_pipeline_eg /bin/bash
```

Then we can `cd` into the `/home/data_analysis_eg` directory and then run `make all` (it is the `/bin/bash` that tells the container to open to bash and not the RStudio GUI).

We can do this in a more automated way. We can link the volume and drive a script or Makefile from a different directory:

```
docker run --rm -v /Users/timberst/Documents/ubc-mds-2017/data_analysis_pipeline_eg:/home/data_analysis_eg ttimbers/data_analysis_pipeline_eg make -C '/home/data_analysis_eg'
```

The general form for this is:

```
docker run --rm -v VOLUME_ON_YOUR_COMPUTER:VOLUME_ON_CONTAINER DOCKER_IMAGE PROGRAM_TO_RUN PROGRAM_ARGUMENTS
```

## To clean up the Makefile

```
docker run --rm -v /Users/timberst/Documents/ubc-mds-2017/data_analysis_pipeline_eg:/home/data_analysis_eg ttimbers/data_analysis_pipeline_eg make -C '/home/data_analysis_eg' clean
```

## Practice!

Let's try this out to get some practice with working with Dockerfiles and Automated builds.

1. Sign-up for an account at [Docker Hub](https://hub.docker.com/)
2. Fork the [data_analysis_pipeline_eg](https://github.com/ttimbers/data_analysis_pipeline_eg) GitHub repo on [GitHub.com](https://github.com/) so that you have your own copy of it. Clone this to your computer.
3. Link your GitHub account to your [Docker Hub](https://hub.docker.com/) account on Docker Hub by clicking "Create" > "Create Automated Build" and following the instructions
4. Follow the rest of the instructions to create an Automated build of the Docker image from the Dockerfile in your forked copy of the data_analysis_pipeline_eg GitHub repo.
5. Go to the [Docker Hub](https://hub.docker.com/) repo you just created and click on "Build Settings". Then click on the "Trigger" button to build your image.
6. Wait 10 min or so, and then you should see the Dockerfile in the [Docker Hub](https://hub.docker.com/) repo.
7. Try pulling and running that image to run the analysis in the data_analysis_pipeline_eg you cloned from [GitHub.com](https://github.com/) (try doing it interactively first, and then non-interactively.
8. Now try modifying the Dockerfile in the data_analysis_pipeline_eg GitHub repo you cloned by adding some other R or Python package. Test that the Docker image still builds and can be run using `docker build`. If it does, commit your Dockerfile changes and push them to Github and see if the Automated build you created updates. If not, trigger it yourself to update.
9. If this all worked, then try to start working on a Dockerfile for your project.

# Concept map wrap-up

- Install R packages inside a Dockerfile
- List good Docker base images for R and Python projects